# Vehicle_stock_fig2.ipynb — Editable figure-generation notebook

Reruns stock regressions from the two original NHGIS CSVs for DIAGNOSTIC FIGURES only. Does not overwrite the WTW numerical results or vehicle-stock projections.

**Run the setup cell first**, then run only the figure cells you want to edit. PNG and SVG are exported together to `results/`. **Original figure colours/line styles are preserved.**

In [ ]:
# First run in this Notebook: locate the extracted project automatically.
# Works when this .ipynb is opened in JupyterLab, Jupyter Notebook or VS Code.
from pathlib import Path
import os, sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
_project = None
for _p in _candidates:
    if (_p / 'code' / 'original_style_notebooks' / 'figure_io.py').is_file() and (_p / 'results').is_dir():
        _project = _p
        break
if _project is None:
    raise FileNotFoundError('Please open this notebook INSIDE the extracted package; keep code/ and results/ together.')
_code_dir = _project / 'code' / 'original_style_notebooks'
(_project / 'results').mkdir(exist_ok=True)
os.chdir(_code_dir)  # Preserve the original Notebook relative paths (no C:\\Users\\... paths).
if str(_code_dir) not in sys.path: sys.path.insert(0,str(_code_dir))
from figure_io import activate_svg_autosave
activate_svg_autosave()  # Each existing fig.savefig('X.png') ALSO saves editable X.svg.
print('PROJECT:',_project)
print('NOTEBOOK CWD:',Path.cwd())
print('OUTPUT:',_project/'results')
# Avoid enormous inline image payloads during batch execution; PNG/SVG are saved
# as actual files. To view in notebook: display(Image(filename=str(_project/'results'/'name.png')))
import matplotlib
matplotlib.use('Agg', force=True)
import matplotlib.pyplot as _plt
_plt.show = lambda *args, **kwargs: _plt.close('all')

# Font/export setting only; original Arial-based sans-serif settings preserved.
import matplotlib as _mpl
_mpl.rcParams['svg.fonttype']='none'
_mpl.rcParams['font.family']='sans-serif'
_mpl.rcParams['font.sans-serif']=['Arial','DejaVu Sans','Helvetica']


## Original SI stock & PCA/Poisson diagnostics — six figures
Source: verified WTW plot code, original Notebook cell 0.

In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from figure_style import style as _publication_style
_publication_style()
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import PoissonRegressor
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial','DejaVu Sans','Helvetica']
plt.rcParams['axes.unicode_minus'] = False

from pathlib import Path
from figure_io import activate_svg_autosave
activate_svg_autosave()
PACKAGE_ROOT = Path.cwd().parents[1]
OUTPUT_DIR = str(PACKAGE_ROOT / 'results')
NYMA_feature_file = str(PACKAGE_ROOT / 'code' / 'original_inputs' / 'nhgis0001_ds272_20245_tract.csv')
NYMA_veh_file = str(PACKAGE_ROOT / 'code' / 'original_inputs' / 'nhgis0002_ds272_20245_tract.csv')

# Consistent color mapping.
COLOR_ICEV = '#d62728'
COLOR_BEV  = '#ffcc00'   # Original explanatory note (plotting or model configuration).
COLOR_PHEV = '#1f77b4'

# ======================== 1. Shanghai model ========================
household_data = {
    'Huangpu':   {'total_hh': 19612, 'no_car': 14066, 'high_value': 835},
    'Xuhui':     {'total_hh': 38779, 'no_car': 24429, 'high_value': 1479},
    'Changning': {'total_hh': 24644, 'no_car': 16158, 'high_value': 1147},
    'Jingan':    {'total_hh': 33771, 'no_car': 22368, 'high_value': 1254},
    'Putuo':     {'total_hh': 45404, 'no_car': 28692, 'high_value': 1750},
    'Hongkou':   {'total_hh': 27696, 'no_car': 18898, 'high_value': 743},
    'Yangpu':    {'total_hh': 42872, 'no_car': 27690, 'high_value': 1231},
    'Minhang':   {'total_hh': 91187, 'no_car': 48431, 'high_value': 4053},
    'Baoshan':   {'total_hh': 80716, 'no_car': 46250, 'high_value': 1934},
    'Jiading':   {'total_hh': 66666, 'no_car': 34161, 'high_value': 1745},
    'Pudong':    {'total_hh': 208926, 'no_car': 115322, 'high_value': 6102},
    'Jinshan':   {'total_hh': 29970, 'no_car': 13720, 'high_value': 614},
    'Songjiang': {'total_hh': 68046, 'no_car': 34042, 'high_value': 2317},
    'Qingpu':    {'total_hh': 50156, 'no_car': 26480, 'high_value': 1493},
    'Fengxian':  {'total_hh': 45007, 'no_car': 23877, 'high_value': 939},
    'Chongming': {'total_hh': 28280, 'no_car': 18715, 'high_value': 264},
}
df_hh = pd.DataFrame(household_data).T
df_hh['car_hh'] = df_hh['total_hh'] - df_hh['no_car']
total_car_hh = df_hh['car_hh'].sum()
df_hh['hh_share'] = df_hh['car_hh'] / total_car_hh

pop_data = {
    'Huangpu':   {'pop': 50.34,  'density': 24604},
    'Xuhui':     {'pop': 109.93, 'density': 20075},
    'Changning': {'pop': 68.53,  'density': 17893},
    'Jingan':    {'pop': 92.93,  'density': 25198},
    'Putuo':     {'pop': 124.87, 'density': 22774},
    'Hongkou':   {'pop': 67.99,  'density': 28957},
    'Yangpu':    {'pop': 119.97, 'density': 19755},
    'Minhang':   {'pop': 272.50, 'density': 7350},
    'Baoshan':   {'pop': 226.39, 'density': 8354},
    'Jiading':   {'pop': 189.04, 'density': 4072},
    'Pudong':    {'pop': 578.58, 'density': 4780},
    'Jinshan':   {'pop': 81.23,  'density': 1386},
    'Songjiang': {'pop': 195.89, 'density': 3234},
    'Qingpu':    {'pop': 128.77, 'density': 1922},
    'Fengxian':  {'pop': 113.95, 'density': 1658},
    'Chongming': {'pop': 59.35,  'density': 501},
}
budget = {
    'Pudong': 1308.66, 'Huangpu': 299.40, 'Xuhui': 281.87, 'Changning': 180.97,
    'Jingan': 302.22, 'Putuo': 166.79, 'Hongkou': 208.82, 'Yangpu': 168.29,
    'Minhang': 336.73, 'Baoshan': 182.70, 'Jiading': 243.42, 'Jinshan': 106.13,
    'Songjiang': 204.55, 'Qingpu': 216.36, 'Fengxian': 216.84, 'Chongming': 94.51,
}
df_pop = pd.DataFrame(pop_data).T
df_budget = pd.Series(budget, name='budget_rev')

df_sh = df_hh.join(df_pop[['pop', 'density']]).join(df_budget)
df_sh['no_car_ratio'] = df_sh['no_car'] / df_sh['total_hh']
df_sh['high_value_ratio'] = df_sh['high_value'] / df_sh['total_hh']
df_sh['budget_per_capita'] = df_sh['budget_rev'] / df_sh['pop']
actual_cars_2024_sh = 613.0   # Original explanatory note (plotting or model configuration).
df_sh['vehicle_stock'] = df_sh['hh_share'] * actual_cars_2024_sh

feature_cols_sh = ['density', 'no_car_ratio', 'high_value_ratio', 'budget_per_capita']
X_sh = df_sh[feature_cols_sh].values
y_sh = df_sh['vehicle_stock'].values

scaler_sh = StandardScaler()
X_scaled_sh = scaler_sh.fit_transform(X_sh)
pca_sh = PCA(n_components=3)
pcs_sh = pca_sh.fit_transform(X_scaled_sh)
pois_sh = PoissonRegressor(alpha=0, max_iter=500)
pois_sh.fit(pcs_sh, y_sh)
y_pred_sh = pois_sh.predict(pcs_sh)
r_sh, p_sh = pearsonr(y_sh, y_pred_sh)

df_sh['predicted'] = y_pred_sh
df_sh = df_sh.reset_index(names='district')

# ======================== 2. NYMA Model ========================
base = pd.read_csv(NYMA_feature_file, dtype={'TRACTA':str,'COUNTYA':str,'STATEA':str})
veh  = pd.read_csv(NYMA_veh_file,      dtype={'TRACTA':str,'COUNTYA':str,'STATEA':str})

base.columns = base.columns.str.strip()
veh.columns  = veh.columns.str.strip()

def make_tract(row):
    return f"{int(row['STATEA']):02d}{int(row['COUNTYA']):03d}{row['TRACTA'].replace('.','').zfill(6)}"

base['tract'] = base.apply(make_tract, axis=1)
veh['tract']  = veh.apply(make_tract, axis=1)

commute_cols = ['AUPWE001','AUPWE003','AUPWE010','AUPWE019']
housing_cols = ['AUVLE001','AUVLE002','AUVLE003']
base = base[['tract'] + commute_cols + housing_cols].copy()

veh_needed = ['tract'] + [f'AUVYE{i:03d}' for i in range(1,8)]
veh = veh[veh_needed].copy()

df_nyma = pd.merge(base, veh, on='tract', how='inner')

msa_fips = [
    '36061','36047','36081','36005','36085',
    '34017','34003','34031',
    '34013','34039',
    '36119','36059','36087','36079','36103',
    '34023','34035','34019','34027','34037','34025','34029'
]

df_nyma['county'] = df_nyma['tract'].str[:5]
df_nyma = df_nyma[df_nyma['county'].isin(msa_fips)].copy()

def to_num(x):
    return pd.to_numeric(str(x).replace(',',''), errors='coerce')
for col in df_nyma.columns:
    if col not in ['tract','county']:
        df_nyma[col] = df_nyma[col].apply(to_num)

core_cols = ['AUVYE001','AUVYE002','AUVYE003','AUVYE004','AUVYE005','AUVYE006','AUVYE007',
             'AUPWE001','AUPWE003','AUPWE010','AUPWE019','AUVLE001','AUVLE002','AUVLE003']
df_nyma = df_nyma.dropna(subset=core_cols)

df_nyma['households']    = df_nyma['AUVYE001']
df_nyma['vehicle_stock'] = (df_nyma['AUVYE003']*1 + df_nyma['AUVYE004']*2 + 
                            df_nyma['AUVYE005']*3 + df_nyma['AUVYE006']*4 + df_nyma['AUVYE007']*5.2)

df_nyma['workers']       = df_nyma['AUPWE001']
df_nyma['drive_share']   = df_nyma['AUPWE003'] / df_nyma['workers']
df_nyma['transit_share'] = df_nyma['AUPWE010'] / df_nyma['workers']
df_nyma['walk_share']    = df_nyma['AUPWE019'] / df_nyma['workers']
df_nyma['total_units']   = df_nyma['AUVLE001']
df_nyma['sf_share']      = (df_nyma['AUVLE002'] + df_nyma['AUVLE003']) / df_nyma['total_units']

feat_cols_nyma = ['drive_share','transit_share','walk_share','sf_share']
df_nyma = df_nyma.replace([np.inf,-np.inf], np.nan).dropna(subset=feat_cols_nyma+['vehicle_stock','households'])

X_nyma = StandardScaler().fit_transform(df_nyma[feat_cols_nyma])
pca_nyma = PCA(n_components=3)
pcs_nyma = pca_nyma.fit_transform(X_nyma)
pois_nyma = PoissonRegressor(alpha=0, max_iter=500)
pois_nyma.fit(pcs_nyma, df_nyma['vehicle_stock'])
nyma_pred = pois_nyma.predict(pcs_nyma)
r_nyma, p_nyma = pearsonr(df_nyma['vehicle_stock'], nyma_pred)

# ======================== 3. Charts ========================
fig1, axes = plt.subplots(1, 2, figsize=(10, 4))
nyma_pca_var = pca_nyma.explained_variance_ratio_
sh_pca_var   = pca_sh.explained_variance_ratio_
labels = ['PC1', 'PC2', 'PC3']

axes[0].bar(labels, sh_pca_var, color='darkorange', alpha=0.9)
axes[0].set_title('Shanghai PCA Explained Variance')
axes[0].set_ylim(0, 1)

axes[1].bar(labels, nyma_pca_var, color='steelblue', alpha=0.9)
axes[1].set_title('New York metropolitan area PCA Explained Variance')
axes[1].set_ylabel('Variance Ratio')
axes[1].set_ylim(0, 1)
fig1.suptitle('PCA Explained Variance Comparison', fontsize=14)
fig1.tight_layout()
fig1.savefig(str(Path(OUTPUT_DIR)/"PCA_variance_comparison_NYMA.png"), dpi=150)
plt.show()

# Shanghai regression
df_sh[['vehicle_stock','predicted']] = df_sh[['vehicle_stock','predicted']] / 100  # 10k -> million
fig2, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df_sh['vehicle_stock'], df_sh['predicted'], c='darkorange', edgecolors='k', s=80)
ax.plot([0, max(df_sh['vehicle_stock'])], [0, max(df_sh['vehicle_stock'])], '--', color='gray', label='1:1 line')
ax.set_xlabel('Actual stock (million)', fontsize=12)
ax.set_ylabel('Predicted stock (million)', fontsize=12)
ax.set_title('Shanghai Regression: Actual vs Predicted')
ax.text(0.05, 0.95, f'Pearson r = {r_sh:.4f}\np = {p_sh:.4f}', transform=ax.transAxes,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat'))
ax.legend()
fig2.tight_layout()
fig2.savefig(str(Path(OUTPUT_DIR)/"Shanghai_regression_diagnosis.png"), dpi=150)
plt.show()

# NYMA regression
nyma_pred_m = nyma_pred / 1e6
df_nyma_m = df_nyma['vehicle_stock'] / 1e6
fig3, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df_nyma_m, nyma_pred_m, c='steelblue', edgecolors='k', s=5, alpha=0.5)
ax.plot([0, df_nyma_m.max()], [0, df_nyma_m.max()], '--', color='gray', label='1:1 line')
ax.set_xlabel('Actual stock (million)', fontsize=12)
ax.set_ylabel('Predicted stock (million)', fontsize=12)
ax.set_title('New York metropolitan area Regression: Actual vs Predicted')
ax.text(0.05, 0.95, f'Pearson r = {r_nyma:.4f}\np = {p_nyma:.4f}', transform=ax.transAxes,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue'))
ax.legend()
fig3.tight_layout()
fig3.savefig(str(Path(OUTPUT_DIR)/"NYMA_regression_diagnosis.png"), dpi=150)
plt.show()

# City-specific input configuration.
sh = pd.read_csv(str(Path(OUTPUT_DIR)/"Shanghai_district_vehicle_stock_2024_2060.csv"))
sh['total_vehicles'] = sh['total_vehicles'] / 100  # 10k -> million

fig4a, ax4a = plt.subplots(figsize=(14, 8))
for d in sorted(sh['district'].unique()):
    sub = sh[sh['district'] == d]
    ax4a.plot(sub['year'], sub['total_vehicles'], marker='.', markersize=3, linewidth=1, label=d)
ax4a.set_title('Shanghai District Total Passenger Car Stock (All Districts)')
ax4a.set_xlabel('Year')
ax4a.set_ylabel('Total vehicles (million)')
ax4a.legend(loc='upper left', bbox_to_anchor=(0.045, 0.83), fontsize=7, ncol=3, framealpha=0.88, facecolor='white')
ax4a.grid(True, alpha=0.3)
fig4a.tight_layout()
fig4a.savefig(str(Path(OUTPUT_DIR)/"Shanghai_district_stock_trend.png"), dpi=150, bbox_inches='tight')
plt.show()

# City-specific input configuration.
sh_total = sh.groupby('year')[['BEV', 'PHEV', 'ICEV']].sum()
sh_total['total'] = sh_total.sum(axis=1)
sh_total[['total','ICEV','BEV','PHEV']] = sh_total[['total','ICEV','BEV','PHEV']] / 100

fig4b, ax4b = plt.subplots(figsize=(8, 5))
ax4b.plot(sh_total.index, sh_total['total'], 'k-', linewidth=2, label='Total stock')
ax4b.stackplot(sh_total.index, sh_total['ICEV'], sh_total['BEV'], sh_total['PHEV'],
               labels=['ICEV', 'BEV', 'PHEV'],
               colors=[COLOR_ICEV, COLOR_BEV, COLOR_PHEV], alpha=0.8)
ax4b.set_title('Shanghai Powertrain Mix 2024-2060')
ax4b.set_xlabel('Year')
ax4b.set_ylabel('Vehicle stock (million)')
ax4b.legend(loc='upper right', bbox_to_anchor=(0.90,0.79), fontsize=9, ncol=1, frameon=True, facecolor='white', framealpha=0.86)
ax4b.grid(True, alpha=0.3)
fig4b.tight_layout()
fig4b.savefig(str(Path(OUTPUT_DIR)/"Shanghai_powertrain_stock.png"), dpi=150, bbox_inches='tight')
plt.show()

# NYMA total stock mix
nyma_all = pd.read_csv(str(Path(OUTPUT_DIR)/"NYMA_tract_vehicle_stock_2024_2060.csv"))
nyma_total = nyma_all.groupby('year')[['total_vehicles', 'BEV', 'PHEV', 'ICEV']].sum()
nyma_total[['total_vehicles','ICEV','BEV','PHEV']] = nyma_total[['total_vehicles','ICEV','BEV','PHEV']] / 1e6

fig5, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(nyma_total.index, nyma_total['total_vehicles'], 'k-', linewidth=2, label='Total stock')
ax1.fill_between(nyma_total.index, 0, nyma_total['ICEV'], alpha=0.8,
                 label='ICEV', color=COLOR_ICEV)
ax1.fill_between(nyma_total.index, nyma_total['ICEV'], nyma_total['ICEV']+nyma_total['BEV'],
                 alpha=0.8, label='BEV', color=COLOR_BEV)
ax1.fill_between(nyma_total.index, nyma_total['ICEV']+nyma_total['BEV'],
                 nyma_total['total_vehicles'], alpha=0.8, label='PHEV', color=COLOR_PHEV)
ax1.set_title('New York metropolitan area Powertrain Mix 2024-2060')
ax1.set_xlabel('Year')
ax1.set_ylabel('Vehicle stock (million)')
ax1.legend(loc='upper right', bbox_to_anchor=(0.90,0.79), fontsize=9, ncol=1, frameon=True, facecolor='white', framealpha=0.86)
ax1.grid(True, alpha=0.3)
fig5.tight_layout()
fig5.savefig(str(Path(OUTPUT_DIR)/"NYMA_stock_prediction.png"), dpi=150, bbox_inches='tight')
plt.show()

# Original explanatory note (plotting or model configuration).
print("\n=== SI Note 3 key numerical results ===")
# Regression model and diagnostics.
print("PCA explained variance ratios:")
print(f"  Shanghai: PC1={sh_pca_var[0]:.4f}, PC2={sh_pca_var[1]:.4f}, PC3={sh_pca_var[2]:.4f}, Cumulative={sh_pca_var.sum():.4f}")
print(f"  NYMA:     PC1={nyma_pca_var[0]:.4f}, PC2={nyma_pca_var[1]:.4f}, PC3={nyma_pca_var[2]:.4f}, Cumulative={nyma_pca_var.sum():.4f}")

# Regression model and diagnostics.
print("\nPoisson regression coefficients:")
print(f"  Shanghai: Intercept={pois_sh.intercept_:.3f}, PC1={pois_sh.coef_[0]:.3f}, PC2={pois_sh.coef_[1]:.3f}, PC3={pois_sh.coef_[2]:.3f}")
print(f"  NYMA:     Intercept={pois_nyma.intercept_:.3f}, PC1={pois_nyma.coef_[0]:.3f}, PC2={pois_nyma.coef_[1]:.3f}, PC3={pois_nyma.coef_[2]:.3f}")

# Original explanatory note (plotting or model configuration).
print(f"\nPearson correlation (Shanghai): r = {r_sh:.4f}, p = {p_sh:.4f}")
print(f"Pearson correlation (NYMA):     r = {r_nyma:.4f}, p = {p_nyma:.4f}")

# Original explanatory note (plotting or model configuration).
sh_2024_pred = df_sh['predicted'].sum()          # Original explanatory note (plotting or model configuration).
nyma_2024_pred = nyma_pred.sum() / 1e6          # Original explanatory note (plotting or model configuration).
print(f"\n2024 total predicted stock (million): Shanghai = {sh_2024_pred:.2f}, NYMA = {nyma_2024_pred:.2f}")

# Original explanatory note (plotting or model configuration).
print("\nAll plots saved to", OUTPUT_DIR)
# Font/export setting only; original Arial-based sans-serif settings preserved.
import matplotlib as _mpl
_mpl.rcParams['svg.fonttype']='none'
_mpl.rcParams['font.family']='sans-serif'
_mpl.rcParams['font.sans-serif']=['Arial','DejaVu Sans','Helvetica']


## NYMA county stock plot — one figure
Source: verified WTW plot code, original Notebook cell 1.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
OUTPUT_DIR = str(_project/'results')
from figure_io import activate_svg_autosave
activate_svg_autosave()
# ======================== NEW: NYMA District Total Stock (All Districts) ========================
nyma_district = pd.read_csv(str(Path(OUTPUT_DIR)/"NYMA_district_vehicle_stock_2024_2060.csv"))

# Original explanatory note (plotting or model configuration).
nyma_district_plt = nyma_district.copy()
nyma_district_plt['total_vehicles'] = nyma_district_plt['total_vehicles'] / 1e6   # Original explanatory note (plotting or model configuration).

fig6, ax = plt.subplots(figsize=(14, 8))
# Legend and label placement.
for d in sorted(nyma_district_plt['DISTRICT'].unique()):
    sub = nyma_district_plt[nyma_district_plt['DISTRICT'] == d]
    ax.plot(sub['year'], sub['total_vehicles'], marker='.', markersize=3, linewidth=1, label=d)

ax.set_title('New York metropolitan area District Total Passenger Car Stock (All Districts)')
ax.set_xlabel('Year')
ax.set_ylabel('Total vehicles (million)')
# Legend and label placement.
ax.legend(loc='upper left', bbox_to_anchor=(0.045, 0.82), fontsize=7, ncol=3, framealpha=0.88, facecolor='white')
ax.grid(True, alpha=0.3)
fig6.tight_layout()
fig6.savefig(str(Path(OUTPUT_DIR)/"NYMA_district_stock_prediction.png"), dpi=150, bbox_inches='tight')
plt.show()
# Font/export setting only; original Arial-based sans-serif settings preserved.
import matplotlib as _mpl
_mpl.rcParams['svg.fonttype']='none'
_mpl.rcParams['font.family']='sans-serif'
_mpl.rcParams['font.sans-serif']=['Arial','DejaVu Sans','Helvetica']
